In [1]:
import joblib
import pandas as pd

# ✅ Load model and features separately
model = joblib.load("loan_model_clean.sav")
features = joblib.load("features.pkl")


In [2]:
features

['ApplicantIncome',
 'CoapplicantIncome',
 'Education_Not Graduate',
 'Self_Employed_Yes',
 'Credit_History_No_History']

In [3]:
# -----------------------------
# Get user input
# -----------------------------
def get_user_input():
    """
    Collect raw loan application details from user input.
    Returns a dictionary usable as raw_input.
    """
    raw_input = {}

    raw_input["ApplicantIncome"] = float(input("Enter Applicant Income: "))
    raw_input["CoapplicantIncome"] = float(input("Enter Coapplicant Income: "))
    raw_input["LoanAmount"] = float(input("Enter Loan Amount: "))
    raw_input["Loan_Amount_Term"] = int(input("Enter Loan Amount Term (months): "))
    raw_input["Credit_History"] = int(input("Credit History (1 = Yes, 0 = No): "))

    raw_input["Gender"] = input("Gender (Male/Female): ").strip()
    raw_input["Education"] = input("Education (Graduate/Not Graduate): ").strip()
    raw_input["Married"] = input("Married (Yes/No): ").strip()
    raw_input["Dependents"] = input("Dependents (0/1/2/3+): ").strip()
    raw_input["Property_Area"] = input("Property Area (Urban/Semiurban/Rural): ").strip()

    return raw_input


In [4]:
## Builds model input (IGNORES extra fields after input for Prediction)

def build_model_input(raw_input, feature_names):

    X = pd.DataFrame(0, index=[0], columns=feature_names)

    # ✅ mapping (MUST MATCH TRAINING)
    gender_map = {'Male': 1, 'Female': 0}
    education_map = {'Graduate': 1, 'Not Graduate': 0}
    married_map = {'Yes': 1, 'No': 0}
    area_map = {'Urban': 2, 'Semiurban': 1, 'Rural': 0}
    credit_map = {1: 1, 0: 0}

    for key, value in raw_input.items():

        if key == "Gender":
            value = gender_map.get(value, 0)

        elif key == "Education":
            value = education_map.get(value, 0)

        elif key == "Married":
            value = married_map.get(value, 0)

        elif key == "Property Area":
            value = area_map.get(value, 0)

        elif key == "Dependents":
            value = 3 if value == '3+' else int(value)

        elif key == "Credit History":
            value = int(value)

        if key in X.columns:
            X.loc[0, key] = value

    return X

In [5]:
raw_input = get_user_input()   # includes Gender, Education, etc.

Enter Applicant Income:  90000
Enter Coapplicant Income:  90000
Enter Loan Amount:  10000
Enter Loan Amount Term (months):  48
Credit History (1 = Yes, 0 = No):  0
Gender (Male/Female):  Male
Education (Graduate/Not Graduate):  Graduate
Married (Yes/No):  Yes
Dependents (0/1/2/3+):  1
Property Area (Urban/Semiurban/Rural):  Urban


In [6]:
print(type(model))
print(hasattr(model, "predict_proba"))

<class 'sklearn.pipeline.Pipeline'>
True


In [7]:
### Predicts the model by using only the best 5 features
X_new = build_model_input(raw_input, features)

prediction = model.predict(X_new)[0]
probability = model.predict_proba(X_new)[0, 1]

print("Prediction (1=Approved, 0=Rejected):", prediction)
print("Approval Probability:", round(probability, 3))

Prediction (1=Approved, 0=Rejected): N
Approval Probability: 0.24


In [8]:
print(X_new)

   ApplicantIncome  CoapplicantIncome  Education_Not Graduate  \
0            90000              90000                       0   

   Self_Employed_Yes  Credit_History_No_History  
0                  0                          0  


#### 
<span style="color: green; font-weight: bold;">
✅ Conclusion: The loan can be approved with a probability of 71%.
</span>

In [9]:
## Approval Table with Threshold
confidence_table = pd.DataFrame({
    "Probability Range": ["0.50 – 0.60", "0.60 – 0.75", "> 0.75"],
    "Interpretation": [
        "Low confidence approval",
        "Medium confidence approval",
        "High confidence approval"
    ]
})

confidence_table

,Probability Range,Interpretation
0,0.50 – 0.60,Low confidence approval
1,0.60 – 0.75,Medium confidence approval
2,> 0.75,High confidence approval
